# 38. The month that ends a day early

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** between, timestamps, date range, half-open interval, month end

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("038-between-on-timestamps")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "payments": """
UEFSMRUEFfAkFdYSTBXOBBUAEgAAuBIMkV8BAAEBAJINCACTDQgAlA0IAJUNCACWDQgAlw0IAJgNCACZDQgAmg0IAJsNCACcDQgA
nQ0IAJ4NCACfDQgAoA0IAKENCACiDQgAow0IAKQNCAClDQgApg0IAKcNCACoDQgAqQ0IAKoNCACrDQgArA0IAK0NCACuDQgArw0I
ALANCACxDQgAsg0IALMNCAC0DQgAtQ0IALYNCAC3DQgAuA0IALkNCAC6DQgAuw0IALwNCAC9DQgAvg0IAL8NCADADQgAwQ0IAMIN
CADDDQgAxA0IAMUNCADGDQgAxw0IAMgNCADJDQgAyg0IAMsNCADMDQgAzQ0IAM4NCADPDQgA0A0IANENCADSDQgA0w0IANQNCADV
DQgA1g0IANcNCADYDQgA2Q0IANoNCADbDQgA3A0IAN0NCADeDQgA3w0IAOANCADhDQgA4g0IAOMNCADkDQgA5Q0IAOYNCADnDQgA
6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPINCADzDQgA9A0IAPUNCAD2DQgA9w0IAPgNCAD5DQgA+g0I
APsNCAD8DQgA/Q0IAP4NCAD/DQgEAGBpeAABDQgAAg0IAAMNCAAEDQgABQ0IAAYNCAAHDQgACA0IAAkNCAAKDQgACw0IAAwNCAAN
DQgADg0IAA8NCAAQDQgAEQ0IABINCAATDQgAFA0IABUNCAAWDQgAFw0IABgNCAAZDQgAGg0IABsNCAAcDQgAHQ0IAB4NCAAfDQgA
IA0IACENCAAiDQgAIw0IACQNCAAlDQgAJg0IACcNCAAoDQgAKQ0IACoNCAArDQgALA0IAC0NCAAuDQgALw0IADANCAAxDQgAMg0I
ADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUN
CABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgAUA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABY
DQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0IAGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgA
aw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0I
AH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJAN
CACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACj
DQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQg8
tmABAAAAAAC3YAEAAAAAABUAFawFFbYFLBXOBBUQFQYVBhwYCLdgAQAAAAAAGAiRXwEAAAAAABYAKAi3YAEAAAAAABgIkV8BAAAA
AAAREQAAANYC9FUBAwAAAM4EAQlLAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGihYsX
MGLImEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3cOLI
mUOnjp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmDJp2sSpk6dPoEKJGkWqlKlTqFKpWsWqlatXsGLJmkWr
lq1buHLp2sWrl69fwIIJG0asmLFjyJIpW8asmbNn0KJJm0atmrVr2LJp28atm7dv4MKJG0eunLlz6NKpW8eunbt38OLJm0evnr17
+PLp28evn79/AAMKHEiwoMGDCBMqXMiwocOHECNKnEixosWLGDNq3Mixo8ePIEOKHEmypEkAFQQV8CQVqiRMFc4EFQASAAC4EvQ0
AQDRHNZjMQYAwIE5CLAvBgAAwOwZmDEGAACY0Z7UMAYAwFDUekkyBgAAAM6TvDAGAAD776mcMQYAgBbevyMuBgCAT7f8OjEGAAAg
xDesMQYAAH3GqmwwBgAAxJujtTEGAAD0WIgRMQYAAM1w0sMvBgAAj/w5WDEGAMBmVbv3LgYAQNAwK3MuBgDA5c5JAi8GAICYeAdM
LgYAQKgh3wkvBgDADAxC3S4GAECV9RDMLgYAQHhnGTkyBgCA68amITEGAIA2nOvqLwYAwMCfCsswBgBA2/JhozEGAEDtRhNWMAYA
AJH2w0wvBgCAbD8nfjIGAMAlCw5cMAYAAHqUINYwBgAAaneRtTAGAMBfgB6GMgYAQAU3BSMyBgAAKjOlvzIGAEBqtoOeMAYAwB+c
RisvBgAArOqSWAF49JsBeYytaS8GAEBoJcBeMAYAQNOLPpkyBgCAV64mtzEGAAD1vK2NLwYAAP037swxBgDAPaCaPC8GAADf45/q
LgYAwGdMKmUyBgDAjBwLGTAGAECRI6gRMgYAgCE9hIIyBgAAA0n3xDIGAAAfeFmALgYAABA9hYouBgDAbJmRvDAGAMBo3B9DMgYA
wN20N6wxBgDAsNLBti4GAEAJO+NHMQYAQOZt0sUxBgAAOI53jC8GAABZzbYoMAYAwLXb2+0uBgCAR/Q2mTEGAAD4yGYtMQYAAAKh
6xoyBgAAt07MpDIGAAATy/BjMAYAQDpYiKwuBgBArBfeDzAGAMA8jPE6MAYAQIGHECkxBgBAIuDObS8GAIARl9CjLwYAAJYh4xQx
BgDAz3INzy4GAMBApyHtLgYAQKau1p4wBgBAgaN57jEGAMDY0DynLgYAQAf/fEwwBgCAg5Re7zEGAIDp+75HLgYAwOZ7h5sxBgBA
is6TES8GAICtCRqNLwYAgC8OWIUxBgAAyptLZC8GAIAFACMHMQYAwOaDLY4xBgCA9QoajCFADCD1T8pBOLAYKhBmLgYAgEBr4rIu
BgBAUWVerzEGAEAALV0bMgYAQMNG82QvBgCAGaT9hzEhGAjbd0Uh0AyBhQ8/ASDwi666LcowBgDA3wzR/jEGAACMHrWjMgYAwHsX
N3wvBgDABYXhUDEGAACXzJgwMQYAgKDzhjYyBgCAcgaTxS8GAEATuX6fLgYAwLy+UyswBgCABpJYejEGAACyhwfhMQYAgM3QMS8v
BgBAsDvJGDIGAMAT4ikhMAYAgHUfCZAyBgBA87Q6PDEGAIAQKlytAYj0UwFqtP4IMQYAAKBIKXswBgAAKQmf8y8GAEBgZ6UYLwYA
AHcagQwwBgBA3ptgiTIGAAAZVQz2LgYAQIXKfpgvBgCA6xFLDzEGAECIDNynMgYAQCJW9JcyBgAANxw9gzEGAICn5JWqLwYAACp9
c/QwBgAAkwfoti4GAMDhvTVGMgYAwKmWq7UuBgAANkUapTIGAIBhyRCSLgYAQDZQRRQvBgAArh1i8S8GAEA+4IXRLwYAQDKgY5Qw
BgCAm8XYszEGAACGuuyxLgYAwD7sDIwvBgBAJ6qgQjIGAIDhREO3MAYAgIuqbnExBgDA5cUnuS8GAABHPlxILwYAQFpKMwUyBgCA
dD5nHC8GAEC0KwhfMQYAgEUtorguBgAAL5P5MDAGAAChI/B3MQYAwKx68CkvBgCA1Vm0LDIGAMC27obvMQYAwJXOACgvBgAAd06m
Oy8GAMAwkzupAfhMK1T3by4GAAB+PmaKLgYAgEK5sDQhUCyh/PeiLwYAgLIiU3gBGAgYaPthqFBAposfSS4GAACftk1qMgYAwNT6
Ap0BEPBSFq3RTi8GAMD5s07FMQYAQAio4FgvBgBAteguPjEGAMBsL/6LMQYAQIk7YJ4yBgCA2EBbKjAGAAClta8wMgYAgCCyA5Iv
BgAA7YDD1zEGAAB9n80BUFAAvKFRcS4GAEBKz4IMLwYAQOkzUyhBAAw5OzfPARgMz0vzBwEQ8Dw47fkDMQYAQIe+tKkuBgDA6lJP
djEGAIAisGcTMQYAAPk+Kp4wBgCAfTtkuDAGAEBfQhDnLwYAQKVB5FEuoSgID+lrAUhsAcYb4TAGAEBadcHRLgYAQPUA/5YyBgAA
IQkm/QEQqGZEm1sxBgDABINY7S8GAACYLKEhMQYAgHFTmlYyBgDAsgz6li8GAIB8XsAlCEjwMqRaLgYAAMgR+qgxBgDAQjvFQagM
AD/o96UYUHf6HrAxBgAATmTJQzAGAIA2mzkmLwGgKB2pzDAGAADle3uRASAsHZegWTIGAEBbY89KARAsdwembDEGAMAmK916ASgw
I0RGEDIGAIATKOUIMoFwiEOxjjIGAMCAkhqnMgYAQCWvE2QxBgAA0pA+iC8GAACAr5o8AQjwW5scYSsuBgBAHupodTIGAMAxI/Lb
LwYAQEROrbcwBgCAqcA8gzEGAMCM99lsMgYAQP6nkscxBgBAApQSOC4GAEA9F5M3MQYAgAn3gqwwBgAAT2zKfjEGAECZHOXxARgM
al3k2QFQCB1pzgH4EEDcA3OtARgMXLy+GAEYSKK17nsuBgDAYLc/MjIGAEBDllUl4Ax3VFSBAYAM/ftEnQEoDDyRO/kBCGziKU/y
LwYAgOhT15EuBgDgfrw3rDEGAAClsKE1AQgQYJscbjIhCAiw6yUBEBCjxuXPL0GwCA4NOAEIcPV8a34vBgBAPCMfYS8GAIBO5K7/
LgYAQO9oR+QwIdhIlf8eMQYAgNRksoUwBgCAlalSJgEIEBBEfKAwQVjwaZaB3y4GAMAIyQncMQYAQHbN4oowBgCAl+YrOzAGAEBP
3ok0MgYAQOEApnIyBgAAo3IfmTAGAMAFCiNFMAYAAEfwH2MwBgAACu+g9jEGAMAdZ4giMgYAwGqUJ5MvBgDAwYSl1TEGAEBRmBBB
QAzAXUpUQVjwh4BLvFLZMQYAANG6u/QwBgDASdcYhzIGAEDETa63MgYAAFdQgE0vBgBASLNN6TEGAMC1/9grLwYAALina04wBgAA
VfZIsDIGAAABota6LgYAQPNXNwUyBgCAAX+XZjIGAAB+6IOyLwYAQDYRvgYwBgAAK/e/zDEGAMC3OoQlLwYAAOlGzCkuBgAVABWs
BRW2BSwVzgQVEBUGFQYcGAgAA0n3xDIGABgIgBbevyMuBgAWACgIAANJ98QyBgAYCIAW3r8jLgYAEREAAADWAvRVAQMAAADOBAEJ
SwACCBhAoICBAwgSKFjAoIGDBxAiSJhAoYKFCxgyaNjAoYOHDyBCiBhBooSJEyhSqFjBooWLFzBiyJhBo4aNGzhy6NjBo4ePH0CC
CBlCpIiRI0iSKFnCpImTJ1CiSJlCpYqVK1iyaNnCpYuXL2DCiBlDpoyZM2jSqFnDpo2bN3DiyJlDp46dO3jy6NnDp4+fP4ACCRpE
qJChQ4gSKVrEqJGjR5AiSZpEqZKlS5gyadrEqZOnT6BCiRpFqpSpU6hSqVrFqpWrV7BiyZpFq5atW7hy6drFq5evX8CCCRtGrJix
Y8iSKVvGrJmzZ9CiSZtGrZq1a9iyadvGrZu3b+DCiRtHrpy5c+jSqVvHrp27d/DiyZtHr569e/jy6dvHr5+/fwADChxIsKDBgwgT
KlzIsKHDhxAjSpxIsaLFixgzatzIsaPHjyBDihxJsqRJABUEFYYXFZAXTBXOBBUAEgAAwwv0wgUAAAD/7QAABpq/AAAECAQAAACf
3AAABZI2AAAAejkAAARw5gAABgI6AAADCFIAAAFBwQAAA58QAAAC/lYAAAVvIQAABHmHAAAFtGEAAAI8YgAABa/YAAABw58AAASQ
9gAAAcb9AAAEWXMAAAWw2wAABDxTAAAEYJsAAAdF8AAAAVX0AAAEvIIAAAU+6AAAAhZtAAABwfwAAARd2AAAA052AAABUI4AAAa/
AAAAA42EAAADZE4AAAIyqgAABh9fAAACeKgAAAO0wQAAA8b+AAAFYxoAAAHBxAAAAPiiAAAARksAAAWlqgAABSf3AAABnGsAAACM
GgAAAP/AAAAGXMMAAASugQAAAsxdAAAC8rgAAAQjkAAABQ/AAAADRgUAAAZCxgAABWtQAAAF+AMAAAQjOQAAAcpJAAAB6J8AAAbA
yAAABQuqAAAHlxYAAAV1agAAATKfAAACa/UAAACkpQAAA113AAAHgikAAAON7gAABe56AAAB/FwAAAFgpQAABoRDAAAHS7sAAAND
lgAAALTpAAAGeyUAAAGsQQAAAMWUAAAAraEAAAcWVQAABN4rAAAAGsoAAALMNQAAApt0AAAGPsgAAAJYuQAABXAMAAADq4IAAAMe
BwAAAufdAAAF55cAAAXaYAAABSxWAAACEMcAAAL9dAAAB25vAAAGQ00AAAFBTwAABwpaAAAE+D8AAADgSgAAAZjQAAAGM/UAAAd0
ZAAABK+cAAABtH8AAAZ+ggAAAS7VAAABgTwAAAJN2gAABCpqAAABVpgAAACLTgAABJkdAAAA3AwAAALH7gAAAgyfAAAAIkIAAAKn
fwAABtGxAAAGLScAAAVzJwAAANLJAAAGbRkAAAN3AwAAAf3tAAACC8cAAAcCFgAAAKLWAAAFtn4AAAPgnAAABOf2AAAEEfQAAAMQ
SwAABEw1AAABZLUAAAVpaQAAAHFhAAACdr8AAAViJgAAAJBfAAAA0I4AAAEaegAAAyEWAAAG/bkAAAYNagAABgQAAAAAtvEAAAVZ
IAAABB6bAAAG9xEAAABubwAAAdwUAAADv0IAAAFbLAAAACdVAAADbrUAAAMdnwAAAvjmAAACANYAAAWpUgAAAPGKAAADc6AAAASC
9wAABeoFAAAAlaIAAAZ5swAABjmfAAAHH4oAAAFecwAAAHItAAABIoEAAAdFtwAABGQtAAADIvoAAAPSGAAAAk1EAAACN90AAAIF
MAAAAusgAAAFvTAAAAVV1AAABF25AAAF0TMAAAM+GQAAABfyAAAF9lIAAANEqgAAAfAFAAAFSx8AAACetwAABSlpAAADnRMAAAHV
TQAABE7NAAADsCwAAAAmmwAABWlTAAAD/DIAAAcgqwAABOR3AAABQtMAAACdkQAABYTAAAAAaTIAAAV2lAAAA4D2AAAD4/4AAAR9
XwAAAq+NAAAHlnUAAAFR4wAABsO3AAAEJssAAAWDLwAABVjbAAAB35gAAAJUkQAABiwHAAADJesAAAIJEQAAAoHdAAADdh4AAAVf
CwAABuZLAAAFfq0AAAb+PgAAAFCOAAAHitgAAAJzMAAAAhXTAAAGo/0AAAYeUAAABMioAAAHRIcAAAIyAQAAAVH0AAAE7QwAAAaJ
SAAAB3y7AAAHbJsAAAHftgAABMhjAAAA5NsAAAIHNgAABxxGAAAFZU4AAAQflQAAB4HyAAAEBkAAAAQjQwAABhVUAAAFlOMAAAUI
BgAAAnnsAAABaCIAAAGZfQAAAlu4AAABYi0AAAI18wAAA/YfAAAC6SEAAAEAXwAAAnvgAAAG/lUAAACPqgAAAYtQAAAC36EAAAIz
NAAAAUyLAAAG8e4AAAd7fgAABCIeAAAGYBMAAATVGwAABO35AAAF0rQAAAb2aAAABVtHAAABTIwAAAXm6QAABt/LAAAEsfIAAALi
rQAABQAbAAAAtNMAAAb0EAAABQqlAAAEY1sAAAExThUAFawFFbYFLBXOBBUQFQYVBhwYBQAAB5cWGAUAAAAX8hYAKAUAAAeXFhgF
AAAAF/IREQAAANYC9FUBAwAAAM4EAQlLAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGi
hYsXMGLImEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3
cOLImUOnjp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmDJp2sSpk6dPoEKJGkWqlKlTqFKpWsWqlatXsGLJ
mkWrlq1buHLp2sWrl69fwIIJG0asmLFjyJIpW8asmbNn0KJJm0atmrVr2LJp28atm7dv4MKJG0eunLlz6NKpW8eunbt38OLJm0ev
nr17+PLp28evn79/AAMKHEiwoMGDCBMqXMiwocOHECNKnEixosWLGDNq3Mixo8ePIEOKHEmypEkAFQQZTDUAGAZzY2hlbWEVBgAV
BCUCGApwYXltZW50X2lkABUEJQIYB3BhaWRfYXQlFEyMEhwsAAAAAAAVDhUKFQIYBmFtb3VudCUKFQQVFCxcFQQVFAAAABbOBBkc
GTwmABwVBBk1AAYQGRgKcGF5bWVudF9pZBUCFs4EFsIrFrIZJoATJggcGAi3YAEAAAAAABgIkV8BAAAAAAAWACgIt2ABAAAAAAAY
CJFfAQAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAM4EAAAAJgAcFQQZNQAGEBkYB3BhaWRfYXQVAhbOBBbCKxaGKyaGPia6
GRwYCAADSffEMgYAGAiAFt6/Iy4GABYAKAgAA0n3xDIGABgIgBbevyMuBgAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAzgQAAAAm
ABwVDhk1AAYQGRgGYW1vdW50FQIWzgQWwB0W1B0m8lsmwEQcGAUAAAeXFhgFAAAAF/IWACgFAAAHlxYYBQAAABfyEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmAM4EAAAAFsR0Fs4EJggWjGIAGRwYDEFSUk9XOnNjaGVtYRjYAi8vLy8vL2dBQUFBUUFBQUFBQUFLQUF3
QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFNQUFBQ0lBQUFBUUFBQUFBUUFBQUNVLy8vL0FB
QUJCeEFBQUFBZ0FBQUFCQUFBQUFBQUFBQUdBQUFBWVcxdmRXNTBBQUFJQUF3QUJBQUlBQWdBQUFBS0FBQUFBZ0FBQU16Ly8vOEFB
QUVLRUFBQUFDQUFBQUFFQUFBQUFBQUFBQWNBQUFCd1lXbGtYMkYwQUFBQUJnQUlBQVlBQmdBQUFBQUFBZ0FRQUJRQUNBQUdBQWNB
REFBQUFCQUFFQUFBQUFBQUFRSVFBQUFBSkFBQUFBUUFBQUFBQUFBQUNnQUFBSEJoZVcxbGJuUmZhV1FBQUFnQURBQUlBQWNBQ0FB
QUFBQUFBQUZBQUFBQUFBQUFBQT09ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZPBwAABwAABwAAABEAwAAUEFS
MQ==
""",
}

_data_dir = pathlib.Path("data") / "038-between-on-timestamps"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

payments = spark.read.parquet((_data_dir / "payments.parquet").resolve().as_posix())
payments.createOrReplaceTempView("payments")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'paid_date': 'a86dbdaef144da866e6b0a5a4f1b45dc5b8a2ede747346426f35aae8b13b5fb2',
                         'payments': 'b13700be039cdb9ce6f490fdfdb37999b53a4fd8b5a3ca9943849e5abc4546a4',
                         'total': '47952557fecd419c5d303aa5ff3344cc4debc922dd97a7b698d46e6ff64a6d2c'},
 'columns': [['paid_date', 'date'], ['payments', 'bigint'], ['total', 'decimal(14,2)']],
 'fingerprint': '182d95b1ab6f06c476beb979cdebb38f370532f1491143e62371f64e5279844e',
 'order_matters': False,
 'row_count': 31}

check = make_check(EXPECTED)

## The situation

Finance closes the books every month from a daily payments table. An AI assistant
wrote the query in the answer cell. It returns one row for each of the 31 days of
March, and the days anyone spot-checked were right.

The month total is still short of the bank statement, and it has been short every
month since the query went live.

Find the bug and fix it.

## Input tables

### `payments`

One row per payment. The table holds payments from mid February to mid April 2025.

| Column | Type | Description |
|---|---|---|
| `payment_id` | BIGINT | Unique per payment. |
| `paid_at` | TIMESTAMP_NTZ | When the payment was made, to the microsecond. |
| `amount` | DECIMAL(10,2) | The amount paid. Never NULL. |

## Requirement

Return the daily totals for March 2025.

- A payment belongs to March 2025 if `paid_at` is on or after
  2025-03-01 00:00:00 and before 2025-04-01 00:00:00.
- `paid_date` is the date part of `paid_at`.
- `payments` is the number of payments on that date.
- `total` is the sum of their amounts.
- Payments outside March do not appear, not even as a row for another date.

## Expected output

One row per date in March 2025 that has payments.

| Column | Type | Description |
|---|---|---|
| `paid_date` | DATE | |
| `payments` | BIGINT | |
| `total` | DECIMAL(14,2) | |

In [ ]:
payments.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `payments`.

In [ ]:
query = """
-- Daily payment totals for March 2025
SELECT
    CAST(paid_at AS DATE) AS paid_date,
    COUNT(*) AS payments,
    CAST(SUM(amount) AS DECIMAL(14, 2)) AS total
FROM payments
WHERE paid_at BETWEEN '2025-03-01' AND '2025-03-31'
GROUP BY CAST(paid_at AS DATE)
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)